# Concept Erasure

Can we identify and surgically remove a concept from a model's representations?
This notebook explores tools for finding concept directions, erasing them,
amplifying them, and measuring their impact on predictions.

This notebook covers the `irtk.concept_erasure` module.

## Why This Matters

Concept erasure removes specific information (like gender or sentiment) from representations while preserving other capabilities. This tests what information is linearly accessible and provides a controlled way to study how models encode and use sensitive attributes.

**Key references:**
- [Belinkov (2022) "Probing Classifiers"](https://arxiv.org/abs/2102.12452) — Methodology for probing neural representations

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

import irtk
from irtk import concept_erasure

model = irtk.HookedTransformer.from_pretrained("gpt2")
tokenizer = model.tokenizer
print(f"Model: {model.cfg.n_layers} layers, d_model={model.cfg.d_model}")

## 1. Finding Concept Directions

A concept direction is a linear direction in activation space that
separates two groups of prompts. For example, "positive sentiment" vs
"negative sentiment" prompts.

In [ ]:
# Positive vs negative sentiment
positive_prompts = [
    "I love this movie, it was absolutely",
    "This restaurant is wonderful and the food is",
    "I'm so happy today because everything is",
    "The vacation was perfect and the weather was",
]
negative_prompts = [
    "I hate this movie, it was absolutely",
    "This restaurant is terrible and the food is",
    "I'm so sad today because everything is",
    "The vacation was awful and the weather was",
]

pos_tokens = [model.to_tokens(p) for p in positive_prompts]
neg_tokens = [model.to_tokens(p) for p in negative_prompts]

# Find concept direction at a late layer
hook = "blocks.10.hook_resid_post"
direction = concept_erasure.find_concept_direction(model, pos_tokens, neg_tokens, hook)
print(f"Concept direction shape: {direction.shape}")
print(f"Concept direction norm: {np.linalg.norm(direction):.4f}")

In [ ]:
# Find concept directions at each layer
layer_magnitudes = []
for l in range(model.cfg.n_layers):
    hook = f"blocks.{l}.hook_resid_post"
    d = concept_erasure.find_concept_direction(
        model, pos_tokens, neg_tokens, hook, normalize=False
    )
    layer_magnitudes.append(np.linalg.norm(d))

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(model.cfg.n_layers), layer_magnitudes)
ax.set_xlabel("Layer")
ax.set_ylabel("Concept Direction Magnitude")
ax.set_title("Sentiment Concept Strength by Layer")
plt.tight_layout()
plt.show()

## 2. Erasing a Concept

Projecting out the concept direction should remove the model's ability
to distinguish between the two groups.

In [ ]:
# Compare predictions with and without the sentiment concept
test_prompt = "I love this movie, it was absolutely"
test_tokens = model.to_tokens(test_prompt)

hook = "blocks.10.hook_resid_post"
direction = concept_erasure.find_concept_direction(model, pos_tokens, neg_tokens, hook)

# Clean predictions
clean_logits = model(test_tokens)
clean_probs = jax.nn.softmax(clean_logits[-1])
top_clean = jnp.argsort(clean_probs)[::-1][:10]
print("Clean top predictions:")
for t in top_clean:
    print(f"  {tokenizer.decode([int(t)]):>15s}: {float(clean_probs[t]):.4f}")

# Erased predictions
erased_logits = concept_erasure.erase_concept(model, test_tokens, hook, direction)
erased_probs = jax.nn.softmax(erased_logits[-1])
top_erased = jnp.argsort(erased_probs)[::-1][:10]
print("\nErased top predictions:")
for t in top_erased:
    print(f"  {tokenizer.decode([int(t)]):>15s}: {float(erased_probs[t]):.4f}")

## 3. Amplifying a Concept

We can also amplify the concept direction to push predictions further
in the concept's direction.

In [ ]:
# Sweep amplification strength
test_prompt = "The movie was"
test_tokens = model.to_tokens(test_prompt)

word_pos = tokenizer.encode(" great")[0]
word_neg = tokenizer.encode(" terrible")[0]

alphas = [0, 0.5, 1.0, 1.5, 2.0, 3.0, 5.0]
for alpha in alphas:
    logits = concept_erasure.amplify_concept(
        model, test_tokens, hook, direction, alpha=alpha
    )
    probs = jax.nn.softmax(logits[-1])
    top_pred = int(jnp.argmax(probs))
    print(f"  alpha={alpha:.1f}: top={tokenizer.decode([top_pred]):>10s}  "
          f"P(great)={float(probs[word_pos]):.4f}  "
          f"P(terrible)={float(probs[word_neg]):.4f}")

## 4. Concept Sensitivity

How much do model predictions change when we erase a concept?
This reveals how important a concept direction is for the model's output.

In [ ]:
test_prompt = "I love this movie, it was absolutely"
test_tokens = model.to_tokens(test_prompt)

sens = concept_erasure.concept_sensitivity(
    model, test_tokens, hook, direction
)
print(f"Logit difference L2: {sens['logit_diff_l2']:.4f}")
print(f"Max logit change: {sens['logit_diff_max']:.4f}")
print(f"Top token logit change: {sens['top_token_change']:.4f}")
print(f"KL divergence: {sens['kl_divergence']:.4f}")

print("\nTop promoted tokens (by erasure):")
for tid, change in sens["top_promoted"][:5]:
    print(f"  {tokenizer.decode([tid]):>15s}: {change:+.4f}")

print("\nTop suppressed tokens (by erasure):")
for tid, change in sens["top_suppressed"][:5]:
    print(f"  {tokenizer.decode([tid]):>15s}: {change:+.4f}")

## 5. LEACE: Optimal Concept Erasure

LEACE (LEAst-squares Concept Erasure) finds the optimal linear projection
that removes all linearly-accessible information about a concept.

In [ ]:
# Collect activations for positive and negative prompts
hook = "blocks.10.hook_resid_post"
all_acts = []
all_labels = []

for tokens in pos_tokens:
    _, cache = model.run_with_cache(tokens)
    all_acts.append(np.array(cache[hook][-1]))
    all_labels.append(1)

for tokens in neg_tokens:
    _, cache = model.run_with_cache(tokens)
    all_acts.append(np.array(cache[hook][-1]))
    all_labels.append(0)

acts = np.stack(all_acts)
labels = np.array(all_labels)
print(f"Activations shape: {acts.shape}")
print(f"Labels: {labels}")

# Apply LEACE
result = concept_erasure.leace(acts, labels)
print(f"\nProjection matrix shape: {result['projection'].shape}")
print(f"Concept direction shape: {result['concept_direction'].shape}")
print(f"Explained variance: {result['explained_variance']:.4f}")

In [ ]:
# Verify erasure: project activations and check class separation
P = result["projection"]
erased_acts = acts @ P.T

# Before erasure
pos_mean = acts[labels == 1].mean(0)
neg_mean = acts[labels == 0].mean(0)
sep_before = np.linalg.norm(pos_mean - neg_mean)

# After erasure
pos_mean_e = erased_acts[labels == 1].mean(0)
neg_mean_e = erased_acts[labels == 0].mean(0)
sep_after = np.linalg.norm(pos_mean_e - neg_mean_e)

print(f"Class separation before erasure: {sep_before:.4f}")
print(f"Class separation after erasure:  {sep_after:.4f}")
print(f"Reduction: {(1 - sep_after/sep_before)*100:.1f}%")

## 6. Concept Spectrum

Is a concept captured by a single direction, or distributed across
multiple orthogonal directions?

In [ ]:
spectrum = concept_erasure.concept_spectrum(
    model, pos_tokens, neg_tokens, hook, k=5
)

print("Concept spectrum:")
for i in range(len(spectrum["singular_values"])):
    print(f"  Direction {i}: sv={spectrum['singular_values'][i]:.4f}  "
          f"variance={spectrum['explained_variance'][i]:.4f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(len(spectrum["explained_variance"])), spectrum["explained_variance"])
ax.set_xlabel("Direction Index")
ax.set_ylabel("Explained Variance")
ax.set_title("Concept Spectrum: Variance per Direction")
plt.tight_layout()
plt.show()

## Summary

| Function | Purpose |
|----------|--------|
| `find_concept_direction()` | Extract a concept direction from contrasting prompts |
| `erase_concept()` | Project out a concept direction during forward pass |
| `amplify_concept()` | Scale up a concept's component in activations |
| `concept_sensitivity()` | Measure how much logits change when concept is erased |
| `leace()` | LEAst-squares Concept Erasure (optimal erasure) |
| `concept_spectrum()` | Find multiple concept directions via PCA |